# qnn-edge-agent: Colab GPU Evaluation Runner

Runs the test suite and evaluates `E0` (FP32), `E1` (`W8A8`), `E2` (naive `W4A8`), `E2a` (`W4A16`), `E2b` (`W16A8`), `E4` (3-boundary SmoothQuant `W4A8`, $\alpha=0.5$), and `E4s` (3-boundary SmoothQuant `W4A8`, searched $\alpha^*$), followed by per-scenario (`pc` vs `iot` vs `bfcl`) diagnostics.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Clone repository
import os
GIT_URL = "https://github.com/iamqazolp/qnn-edge-agent.git"
REPO_NAME = "qnn-edge-agent"

if not os.path.exists(REPO_NAME):
    !git clone $GIT_URL $REPO_NAME
%cd $REPO_NAME
!git pull

In [ ]:
# 3. Install dependencies in editable mode
!pip install -e ".[dev]"

In [ ]:
# 4. Run verification tests & inspect recipes
!pytest tests/
!qea recipes

In [ ]:
# 5. Run ablation suite across E0, E1, E2, E2a, E2b, E4, E4s
!python scripts/run_ablation_suite.py --ids E0,E1,E2,E2a,E2b,E4,E4s

In [ ]:
# 6. Summary comparison table + per-scenario (PC vs IoT vs BFCL) diagnostics
from pathlib import Path
import json

print(f"{'Recipe':<8} | {'Avg Bits':<8} | {'Footprint':<10} | {'PPL':<8} | {'Mode':<14} | {'JSON Valid':<11} | {'Exact Val':<11}")
print("-" * 80)
for rid in ["E0", "E1", "E2", "E2a", "E2b", "E4", "E4s"]:
    p = Path(f"results/{rid}.json")
    if not p.exists():
        continue
    with open(p) as f:
        d = json.load(f)
    for mode, sc in d.get("eval", {}).items():
        ppl_str = f"{d.get('wikitext2_ppl', 0.0):.2f}" if d.get('wikitext2_ppl') else "N/A"
        print(f"{rid:<8} | {d['avg_bits_linear']:<8.1f} | {d['footprint_mb']:<7.1f} MB | {ppl_str:<8} | {mode:<14} | {sc['json_valid']['rate']*100:<10.1f}% | {sc['values_exact']['rate']*100:<10.1f}%")

print("\n--- Scenario Breakdown (PC vs IoT vs BFCL) ---")
!qea diagnose --results-dir results